# How to use rugo with Avro

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mabel-dev/rugo.dev/blob/main/docs/notebooks/avro.ipynb)

Read an Avro file, look at its schema, read only the fields you need, filter rows, load it into pandas, and convert it to Parquet - with rugo, which needs no PyArrow, pandas or NumPy of its own. **Rugo reads Avro; it doesn't write it.**

Every example uses the same dataset: 4,630 orbital launch attempts, from Sputnik onwards. More at [rugo.dev](https://rugo.dev).

In [ ]:
%pip install -q rugo pandas

In [ ]:
import os
import urllib.request

if not os.path.exists("space_missions.avro"):
    urllib.request.urlretrieve("https://raw.githubusercontent.com/mabel-dev/rugo.dev/main/docs/data/space_missions.avro", "space_missions.avro")
print(f"space_missions.avro: {os.path.getsize('space_missions.avro'):,} bytes")

## Read a file

`read_avro` is a context manager that yields **morsels** - batches of typed columns, made of whole Avro blocks. Logical types are decoded (the launch time arrives as a timestamp), enums arrive as their symbol, and a nested record arrives as JSON text.

In [ ]:
from rugo import avro

with avro.read_avro("space_missions.avro") as reader:
    for morsel in reader:
        print(morsel)

## Look at the schema without reading the data

An Avro file carries the schema it was written with in its header. `read_metadata` returns it, with the compression codec, without decoding a record. `["null", "double"]` is how Avro spells a nullable column; `Rocket` is a nested record.

In [ ]:
meta = avro.read_metadata("space_missions.avro")
print("codec:", meta.codec)
for field in meta.schema["fields"]:
    print(f"{field['name']:16} {field['type']}")

## Read only some fields

Name the fields you want. A dotted name reads one field out of a nested record as an ordinary column.

In [ ]:
with avro.read_avro(
    "space_missions.avro",
    columns=["Company", "Mission", "Rocket.Name", "Rocket.Status"],
) as reader:
    for morsel in reader:
        print(morsel)

## Filter rows

An Avro file has no statistics a reader could use to skip records, so there's no `predicates` argument: read the fields you need and filter the values.

In [ ]:
with avro.read_avro("space_missions.avro", columns=["Company", "Mission", "Price"]) as reader:
    rows = [
        (company, mission, price)
        for morsel in reader
        for company, mission, price in zip(
            morsel.column("Company").to_pylist(),
            morsel.column("Mission").to_pylist(),
            morsel.column("Price").to_pylist(),
        )
        if company == "SpaceX" and price is not None and price < 60
    ]

print(len(rows), "cheap SpaceX launches")
print(rows[:3])

## Read through a reader schema

A reader schema is the shape you want, matched against the shape the file was written with: fields it leaves out are dropped, fields the file doesn't have take their default, and numbers can widen (int to long, float to double). It's how Avro reads old and new versions of a schema together.

In [ ]:
reader_schema = {
    "type": "record",
    "name": "Launch",
    "fields": [
        {"name": "Mission", "type": "string"},
        {"name": "Price", "type": ["null", "double"], "default": None},
        {"name": "Source", "type": "string", "default": "space_missions"},
    ],
}

with avro.read_avro("space_missions.avro", reader_schema=reader_schema) as reader:
    for morsel in reader:
        print(morsel)

## Load a file into pandas

Rugo doesn't need pandas, but it's easy to hand data over: each column's `to_pylist()` gives the values, and a dict of lists is a DataFrame. This little helper does it for every morsel a reader yields.

In [ ]:
import pandas as pd


# Every morsel a rugo reader yields, as one pandas DataFrame.
def to_pandas(reader):
    frames = [
        pd.DataFrame({name.decode(): morsel.column(name).to_pylist() for name in morsel.column_names})
        for morsel in reader
    ]
    return pd.concat(frames, ignore_index=True)

In [ ]:
with avro.read_avro("space_missions.avro", columns=["Company", "Mission", "Price", "Rocket.Name"]) as reader:
    df = to_pandas(reader)

df[df["Company"] == "SpaceX"].head()

## Convert to Parquet

Rugo doesn't write Avro, but it writes Parquet, CSV and JSON Lines - and readers and writers share the morsel, so converting is a read and a write. The Avro schema's types are carried into the Parquet file.

In [ ]:
from rugo import parquet

with avro.read_avro("space_missions.avro") as reader:
    with open("space_missions.parquet", "wb") as f:
        with parquet.open_parquet_writer(f.write) as writer:
            for morsel in reader:
                writer.write_row_group(morsel)

for column in parquet.read_metadata("space_missions.parquet").schema_columns:
    print(f"{column.name:16} {column.logical_type}")